# Fitting orthogonalised Fourier aberrations to a Zernike state

Finds the coefficients of amigo's orthogonalised Fourier basis whose OPD best matches the Zernike OPD stored in a fitted state
(`final_statev46.npy`), for every aberration key, so a retrain can start from them instead of from zero.

The Fourier basis is the one `AMIOptics(fourier=True)` builds (`orthogonalise_fourier_basis`): an SVD of the 13×13 Fourier
dictionary over the oversized hexagon, keeping the top `N_KEEP` singular vectors. Being orthogonal over the oversized hexagon,
it stays fixed while `distortion` / `primary_beam` are fitted in a retrain.

Both OPDs are built with the pipeline's own objects (`AMIOptics(static=False, sparse=True)` → `DynamicApertureMask`,
`eval_small_basis`), with the state's fitted `distortion` / `primary_beam`, and compared per hole over the **fitted
aperture** (`sparse_apertures`, soft edges as weights) — the only region light actually passes through.

The loss is quadratic in the coefficients, so the optimum is a weighted least-squares solve. The optax loop below is there to
fit in the same way the retrain does; the closed-form solve is computed alongside as a check that the loop reached the optimum.

The two bases span different spaces, so the match is not exact: the SVD keeps the modes with the largest singular values,
which leaves part of several low-order Zernikes unrepresentable. Section 4 shows this per Noll mode.

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)

import jax.numpy as np
import numpy as onp
import optax
from jax import vmap

import amigo
from amigo.optical_models import AMIOptics, eval_small_basis

import matplotlib.pyplot as plt
import matplotlib as mpl
import scienceplots
import os

plt.style.use(["science", "bright", "no-latex"])
plt.rcParams["image.cmap"] = "inferno"
plt.rcParams["font.family"] = "serif"
plt.rcParams["image.origin"] = "lower"
plt.rcParams["font.size"] = 8

seismic = mpl.colormaps["seismic"]
seismic.set_bad("k", 0.5)

load_dict = lambda x: onp.load(f"{x}", allow_pickle=True).item()

print(jax.local_devices()[0].device_kind, "| amigo", amigo.__version__)

In [ ]:
from socket import gethostname

if gethostname() in ("maxs-mbp-14.shared.sydney.edu.au", "Maxs-MacBook-Pro-14.local"):
    amigo_files_path = "/Volumes/research-data/PRJ-PAT/max/data/amigo_files/v_0.0.11"
elif gethostname().startswith("max-"):
    amigo_files_path = "/home/dgxuser/max/data/amigo_files/v_0.0.11"
else:
    amigo_files_path = "/fred/oz440/max/data/amigo_files/v_0.0.11"
print(f"amigo_files_path: {amigo_files_path}")

# ---- knobs -----------------------------------------------------------------
STATE_NAME = "final_statev46"
N_KEEP = 28  # Fourier modes per hole (the pipeline's n_modes_keep)
N_DICT = 13  # per-axis dictionary size (the pipeline's n_modes)
RADIAL_ORDERS = 7  # Zernike basis the state was fitted with (7 -> 28 Noll modes)
F2F, OVERSIZE = 0.80, 1.2
DIAMETER, NPIX = 6.603464, 1024

EPOCHS = 2000
LR = optax.exponential_decay(20.0, 500, 0.3)  # nm per step; coefficients are in nm
SAVE = True  # write a copy of the state with the fitted Fourier coefficients next to the state

state = load_dict(os.path.join(amigo_files_path, STATE_NAME + ".npy"))
keys = list(state["aberrations"])
print(f"{len(keys)} aberration keys, each {state['aberrations'][keys[0]].shape}")

## 1. Pipeline optics with the fitted aperture
Two `AMIOptics`, one Zernike (what the state was fitted with) and one Fourier, both with the state's fitted aperture geometry.

In [ ]:
kw = dict(radial_orders=RADIAL_ORDERS, static=False, sparse=True, f2f=F2F, oversize=OVERSIZE)
optics_z = AMIOptics(**kw)
optics_f = AMIOptics(**kw, fourier=True, n_modes_keep=N_KEEP, n_modes=N_DICT)

geometry = ["pupil_mask.transformation.distortion", "pupil_mask.primary_beam"]
values = [np.asarray(state["distortion"]), np.asarray(state["primary_beam"])]
optics_z, optics_f = optics_z.set(geometry, values), optics_f.set(geometry, values)
pm_z, pm_f = optics_z.pupil_mask, optics_f.pupil_mask

# per-hole fitted apertures (n_holes, 180, 180): what the model multiplies the wavefront by
mask = pm_z.sparse_apertures(NPIX, DIAMETER)
n_holes = mask.shape[0]
print("fitted aperture vs the state's saved transmission:",
      float(np.abs(pm_z.calc_mask(NPIX, DIAMETER) - state["transmission"]).max()))

## 2. The Fourier basis
In the pipeline's units: `1e-9 ×` modes with RMS ≈ 1 over the oversized hexagon, so coefficients are in nm.
The condition number printed below is measured over the fitted aperture, i.e. how correlated the coefficients are where light passes.

In [ ]:
basis_f = pm_f.abb_basis
basis_z = pm_z.abb_basis
print("Fourier", basis_f.shape, "| Zernike", basis_z.shape)


def gram(B, m):
    # normalised inner products of the modes over the soft fitted aperture
    b = B.reshape(B.shape[0], -1) * np.sqrt(m.reshape(-1))
    G = b @ b.T
    d = np.sqrt(np.diag(G))
    return G / np.outer(d, d)


conds = [float(np.linalg.cond(gram(basis_f[h], mask[h]))) for h in range(n_holes)]
print("condition number over the fitted aperture, per hole:", onp.round(conds, 3))

## 3. Fitting loop
Target: each key's Zernike OPD in nm, `(n_keys, n_holes, 180, 180)`. Loss: per-hole squared error weighted by the soft
fitted aperture (normalised per hole), averaged over holes and keys, so `sqrt(loss)` is an RMS in nm. All keys are fitted at once.

In [ ]:
C_z = np.stack([np.asarray(state["aberrations"][k]) for k in keys])  # (K, n_holes, 28) nm
opd = lambda B, c: eval_small_basis(B, c) / 1e-9  # nm
target = vmap(lambda c: opd(basis_z, c))(C_z)
w = mask / mask.sum((-1, -2), keepdims=True)

wmse = lambda r: (w * r**2).sum((-1, -2))  # (..., n_holes)
per_key = lambda C: vmap(lambda c, t: wmse(opd(basis_f, c) - t))(C, target)  # (K, n_holes)
loss_fn = lambda C: per_key(C).mean()

optim = optax.adam(LR)


@jax.jit
def step_fn(C, opt_state):
    loss, grads = jax.value_and_grad(loss_fn)(C)
    updates, opt_state = optim.update(grads, opt_state, C)
    return optax.apply_updates(C, updates), opt_state, loss


C_f = np.zeros((len(keys), n_holes, N_KEEP))
opt_state = optim.init(C_f)
losses = []
for i in range(EPOCHS):
    C_f, opt_state, loss = step_fn(C_f, opt_state)
    losses.append(float(loss))
    if i % 250 == 0:
        print(f"epoch {i:5d}  rms residual {onp.sqrt(losses[-1]):8.3f} nm")
print(f"final        rms residual {onp.sqrt(float(loss_fn(C_f))):8.3f} nm  (target rms {onp.sqrt(float(wmse(target).mean())):.1f} nm)")

In [ ]:
# closed-form weighted least squares, per key and hole: the loop should land on this
A = basis_f.reshape(n_holes, N_KEEP, -1) / 1e-9
sw = np.sqrt(w.reshape(n_holes, -1))
lstsq_one = lambda t: vmap(lambda a, s, y: np.linalg.lstsq((a * s).T, y.reshape(-1) * s)[0])(A, sw, t)
C_ls = vmap(lstsq_one)(target)
print(f"loop vs least squares: rms {onp.sqrt(float(loss_fn(C_f))):.6f} vs {onp.sqrt(float(loss_fn(C_ls))):.6f} nm, "
      f"max |coefficient difference| {float(np.abs(C_f - C_ls).max()):.2e} nm")

fig, ax = plt.subplots(figsize=(5, 3))
ax.semilogy(onp.sqrt(losses))
ax.axhline(onp.sqrt(float(loss_fn(C_ls))), color="r", ls="--", label="least squares")
ax.set(xlabel="epoch", ylabel="rms residual (nm)", title=f"Fourier, {N_KEEP} modes")
ax.legend()
plt.show()

## 4. How good is the match?
Per key: RMS of the Zernike OPD and of the residual over the fitted aperture. Per Noll mode: the fraction of each unit
Zernike that the Fourier basis cannot represent on the fitted aperture (worst hole) — the residual floor comes from these.

In [ ]:
res_key = onp.sqrt(onp.asarray(per_key(C_f)).mean(1))
tgt_key = onp.sqrt(onp.asarray(wmse(target)).mean(1))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5), gridspec_kw={"width_ratios": [2, 1]})
x = onp.arange(len(keys))
ax[0].bar(x - 0.2, tgt_key, 0.4, label="Zernike OPD")
ax[0].bar(x + 0.2, res_key, 0.4, label="residual")
ax[0].set_xticks(x, keys, rotation=90, fontsize=5)
ax[0].set(ylabel="rms over fitted aperture (nm)", title="per aberration key")
ax[0].legend()

# unexplained fraction of each unit Zernike mode, per hole
unexpl = []
for h in range(n_holes):
    Q = np.linalg.qr((A[h] * sw[h]).T)[0]
    Zw = (basis_z[h].reshape(basis_z.shape[1], -1) / 1e-9 * sw[h]).T
    Zr = Zw - Q @ (Q.T @ Zw)
    unexpl.append((Zr**2).sum(0) / (Zw**2).sum(0))
unexpl = onp.asarray(unexpl)
ax[1].bar(onp.arange(1, unexpl.shape[1] + 1), unexpl.max(0))
ax[1].set(xlabel="Noll index", ylabel="fraction not representable", ylim=(0, 1), title="worst hole")
plt.tight_layout()
plt.show()
print("worst key:", keys[int(res_key.argmax())], f"{res_key.max():.2f} nm")

In [ ]:
def show_key(key, hole_mask=mask):
    k = keys.index(key)
    fit = onp.asarray(opd(basis_f, C_f[k]))
    tgt = onp.asarray(target[k])
    m = onp.asarray(hole_mask) > 0.5
    rows = {"Zernike (state)": tgt, "Fourier": fit, "residual": fit - tgt}
    v = onp.abs(onp.where(m, tgt, 0)).max()
    fig, ax = plt.subplots(3, n_holes, figsize=(1.8 * n_holes, 5.6))
    for r, (name, arr) in enumerate(rows.items()):
        vr = v if r < 2 else onp.abs(onp.where(m, arr, 0)).max()
        for h in range(n_holes):
            im = ax[r, h].imshow(onp.where(m[h], arr[h], onp.nan), cmap=seismic, vmin=-vr, vmax=vr)
            ax[r, h].set(xticks=[], yticks=[])
            if r == 0:
                ax[r, h].set_title(f"hole {h}")
        ax[r, 0].set_ylabel(name)
        fig.colorbar(im, ax=ax[r], shrink=0.8, label="nm")
    fig.suptitle(f"{key}: OPD over the fitted aperture")
    plt.show()


show_key(keys[int(res_key.argmax())])
show_key("04481_F430M")

All keys: residual RMS per key and hole, then the residual OPD for every key (rows) and hole (columns) on one shared
colour scale (clipped at the 99.5th percentile over all apertures, so a few edge pixels don't wash it out).

In [ ]:
res_kh = onp.sqrt(onp.asarray(per_key(C_f)))  # (K, n_holes) nm

fig, ax = plt.subplots(figsize=(4.5, 0.2 * len(keys) + 1))
im = ax.imshow(res_kh, cmap="inferno", aspect="auto", origin="upper")
for (k, h), v in onp.ndenumerate(res_kh):
    ax.text(h, k, f"{v:.0f}", ha="center", va="center", fontsize=5, color="w" if v < 0.6 * res_kh.max() else "k")
ax.set_yticks(range(len(keys)), keys, fontsize=5)
ax.set(xticks=range(n_holes), xlabel="hole", title="residual rms (nm)")
fig.colorbar(im, ax=ax, label="nm")
plt.show()

resid = onp.asarray(vmap(lambda c: opd(basis_f, c))(C_f) - target)  # (K, n_holes, S, S)
m = onp.asarray(mask) > 0.5
v = onp.percentile(onp.abs(resid[:, m]), 99.5)
fig, ax = plt.subplots(len(keys), n_holes, figsize=(1.0 * n_holes + 2.5, 1.0 * len(keys)), layout="constrained")
for k in range(len(keys)):
    for h in range(n_holes):
        im = ax[k, h].imshow(onp.where(m[h], resid[k, h], onp.nan), cmap=seismic, vmin=-v, vmax=v)
        ax[k, h].set(xticks=[], yticks=[])
        if k == 0:
            ax[k, h].set_title(f"hole {h}")
    ax[k, 0].set_ylabel(keys[k], fontsize=7, rotation=0, ha="right", va="center")
fig.colorbar(im, ax=ax[:, -1], shrink=0.1, label="residual (nm)", location="right")
fig.suptitle("Fourier - Zernike OPD over the fitted aperture, all keys")
plt.show()

## 5. Save
A full copy of the state with only `aberrations` replaced by the fitted Fourier coefficients (`{key: (n_holes, N_KEEP)}`, nm).
The basis itself is rebuilt by `AMIOptics(fourier=True, n_modes_keep=N_KEEP, n_modes=N_DICT)`, so it isn't saved.

Note: `update_optics` stops the gradient of `aberrations[0, 0]` (hole 0's first mode) to remove the piston degeneracy.
In the Zernike basis that is exactly piston; in the Fourier basis mode 0 is only roughly piston, so that coefficient is
frozen at its fitted value here during a retrain.

In [ ]:
ab_fourier = {k: onp.asarray(C_f[i]) for i, k in enumerate(keys)}
out_state = dict(state) | {"aberrations": ab_fourier}

if SAVE:
    out_path = os.path.join(amigo_files_path, f"{STATE_NAME}_fourier_{N_KEEP}.npy")
    onp.save(out_path, out_state, allow_pickle=True)
    print("saved state:", out_path)
else:
    print("SAVE = False, nothing written")